# 04 — Session d'entraînement (Colab T4)

**Objectif** : lancer une session d'entraînement sur GPU, robuste aux coupures Colab et avec visibilité totale sur ce qui se passe.

**Trois améliorations majeures depuis la v1** :

1. **Données copiées sur SSD local** (`/content/data`) avant l'entraînement → ~50× plus rapide que Drive (FUSE).
2. **Print de progression par batch** dans la console → tu vois si l'epoch avance, sans attendre 7 min.
3. **Smoke test 30 s** avant le vrai run → si quelque chose plante, tu le sais en 30 s pas en 20 min.

**Robustesse aux coupures** :
- Sauvegarde de `last.pt` toutes les 3 min (intra-epoch) + à chaque fin d'epoch.
- Log `train_loss_step` à wandb tous les ~10 % de l'epoch → courbe live dès la 1re minute.
- Si Colab coupe : relancer toutes les cellules. La reprise détecte `last.pt` et continue.

## 0. Anti-idle Colab (recommandé pour les longues sessions)

Colab Free coupe les sessions inactives au bout de ~90 min. Pour éviter ça :

1. **Garde cet onglet visible** (premier plan, pas minimisé).
2. **Snippet JS anti-idle** : `F12` → onglet *Console* → colle et entrée :

```js
setInterval(() => {
  const btn = document.querySelector('colab-toolbar-button#connect');
  if (btn) btn.click();
  console.log('[anti-idle] tick', new Date().toLocaleTimeString());
}, 60_000);
```

3. **Pas de mise en veille** du PC (l'écran peut s'éteindre, c'est ok).

## 1. Setup Colab (Drive + repo + dépendances)

In [ ]:
# Montage Drive (Colab uniquement)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab :", IN_COLAB)

In [ ]:
# Clone / pull du repo + cd dedans + install
import os, subprocess
REPO_DIR = "/content/Filtre-Voix-DL" if IN_COLAB else os.path.abspath("..")
REPO_URL = "https://github.com/Theo-Lempereur/Filtre-Voix-DL.git"
BRANCH   = "features/training"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "fetch"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run(["pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("REPO_DIR :", REPO_DIR)

In [ ]:
# Vérification GPU
import torch
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

## 2. Diagnostic (à exécuter au moins la 1re fois)

Trois mesures rapides pour confirmer que tout est en place avant de lancer un long entraînement.

In [ ]:
# 2a. Nombre de paires dans chaque split
from src import config as cfg
for split in ["train", "val", "test"]:
    for kind in ["clean", "noisy"]:
        d = os.path.join(cfg.DRIVE_PROJECT, "data", split, kind)
        n = len(os.listdir(d)) if os.path.isdir(d) else 0
        print(f"  data/{split}/{kind:<6s} : {n:>4} fichiers")

In [ ]:
# 2b. Vitesse de lecture depuis Drive
import time
from src.dataset import PairedAudioDataset

ds_drive = PairedAudioDataset(noisy_dir=cfg.DATA_TRAIN_NOISY,
                              clean_dir=cfg.DATA_TRAIN_CLEAN)
t0 = time.time()
for i in range(min(10, len(ds_drive))):
    _ = ds_drive[i]
elapsed_drive = time.time() - t0
print(f"10 paires depuis Drive : {elapsed_drive:.1f}s ({elapsed_drive*100:.0f} ms/paire)")
if elapsed_drive > 2:
    print("  → IO Drive lent. La cellule de copie sur /content/ (étape 3) va bien aider.")
else:
    print("  → IO Drive rapide, la copie locale reste utile mais moins critique.")

## 3. Copie du dataset sur SSD local (`/content/data`)

Drive est lent. Copier le dataset (~250 MB pour 500 paires) sur le SSD local Colab prend ~30 s et **accélère l'entraînement d'un facteur 10 à 50**.

Les **checkpoints et logs restent sur Drive** (persistance entre sessions). Seules les données d'entrée sont locales (recopiées au prochain démarrage Colab).

In [ ]:
import shutil

LOCAL_DATA = "/content/data"
need_copy = not os.path.isdir(os.path.join(LOCAL_DATA, "train", "clean"))

if need_copy:
    src = os.path.join(cfg.DRIVE_PROJECT, "data")
    print(f"Copie {src} → {LOCAL_DATA} (~30 s)…")
    t0 = time.time()
    shutil.copytree(src, LOCAL_DATA, dirs_exist_ok=True)
    print(f"Terminé en {time.time()-t0:.1f}s")
else:
    print(f"{LOCAL_DATA} déjà présent (rien à copier).")

for split in ["train", "val", "test"]:
    d = f"{LOCAL_DATA}/{split}/clean"
    n = len(os.listdir(d)) if os.path.isdir(d) else 0
    print(f"  /content/data/{split}/clean : {n} fichiers")

In [ ]:
# Bench comparatif pour confirmer le gain
ds_local = PairedAudioDataset(noisy_dir=f"{LOCAL_DATA}/train/noisy",
                              clean_dir=f"{LOCAL_DATA}/train/clean")
t0 = time.time()
for i in range(min(10, len(ds_local))):
    _ = ds_local[i]
elapsed_local = time.time() - t0
print(f"10 paires depuis SSD local : {elapsed_local:.1f}s ({elapsed_local*100:.0f} ms/paire)")
print(f"Speedup vs Drive : ×{elapsed_drive/max(elapsed_local, 0.01):.1f}")

## 4. Wandb (optionnel)

Décommenter pour s'authentifier une fois par session Colab.

In [ ]:
# import wandb; wandb.login()

## 5. Configuration du run

**Config optimale figée** pour ce setup (T4, ~400 paires train, data sur SSD local).

⚠️ Si tu reprends un run d'un jour précédent, **fige le `RUN_ID`** (commenter la ligne `datetime.now()` et décommenter la suivante).

In [ ]:
from datetime import datetime

RUN_TAG = "base16-lr1e3"
RUN_ID  = f"{datetime.now():%Y%m%d}-{RUN_TAG}"
# Pour reprendre un run existant, décommenter et fixer :
# RUN_ID = "20260524-base16-lr1e3"

run_config = {
    "run_id":                    RUN_ID,

    # --- Données pointées sur le SSD local ---
    "train_clean":               f"{LOCAL_DATA}/train/clean",
    "train_noisy":               f"{LOCAL_DATA}/train/noisy",
    "val_clean":                 f"{LOCAL_DATA}/val/clean",
    "val_noisy":                 f"{LOCAL_DATA}/val/noisy",

    # --- Modèle adapté au petit dataset (~400 paires train) ---
    "base_channels":             16,        # ~2 M params : limite l'overfit

    # --- Optimisation ---
    "lr":                        1e-3,
    "weight_decay":              1e-5,      # léger anti-overfit
    "batch_size":                8,
    "num_workers":               2,         # OK avec data sur SSD local
    "num_epochs":                30,        # early stopping coupera avant
    "seed":                      42,
    "grad_clip_norm":            1.0,

    # --- Régulation ---
    "early_stop_patience":       5,
    "lr_patience":               3,

    # --- Anti-coupure ---
    "intra_epoch_save_seconds":  3 * 60,    # save toutes les 3 min
    "intra_epoch_log_every":     "auto",    # ~10 points train_loss_step / epoch
    "ckpt_every_n_epochs":       1,
    "keep_last_n_ckpt":          3,

    # --- Wandb ---
    "use_wandb":                 True,
    "notes":                     "Optimal : base=16, SSD local, log auto.",
}
print("RUN_ID :", RUN_ID)

## 6. Reprise automatique sur checkpoint

In [ ]:
from pathlib import Path
from src.checkpoint import find_latest_checkpoint

ckpt_dir = Path(cfg.CHECKPOINTS) / RUN_ID
resume_from = find_latest_checkpoint(ckpt_dir)

if resume_from is None:
    print(f"[reprise] aucun checkpoint dans {ckpt_dir} → entraînement neuf.")
    ckpt_root = Path(cfg.CHECKPOINTS)
    if ckpt_root.exists():
        others = sorted(p.name for p in ckpt_root.iterdir() if p.is_dir())
        if others:
            print("[reprise] autres runs disponibles (mets son nom dans RUN_ID pour reprendre) :")
            for name in others:
                print(f"           - {name}")
else:
    print(f"[reprise] checkpoint détecté : {resume_from}")

## 7. Smoke test (< 90 s) — valide tout le pipeline

Lance un mini-entraînement sur 16 paires / 1 epoch. Si ça termine avec `best.pt` + `last.pt` créés, le pipeline est sain à 100 %. Sinon on debug en 30 s au lieu de perdre 20 min sur un vrai run.

In [ ]:
from src.train import train

print("=" * 60)
print("SMOKE TEST — devrait terminer en < 90 s")
print("=" * 60)
smoke_cfg = {**run_config,
             "run_id":              f"smoke-{int(time.time())}",
             "num_epochs":          1,
             "use_wandb":           False,
             "early_stop_enabled":  False}
smoke_t0 = time.time()
_ = train(smoke_cfg, resume_from=None,
          max_train_samples=16, max_val_samples=8)
print(f"\nSMOKE TEST OK ({time.time()-smoke_t0:.1f}s)")
print("=" * 60)

## 8. Vrai entraînement

Pendant l'exécution tu vois dans la console :
- Une ligne `ep N | batch X/Y | loss Z | T s/batch` toutes les 5 batches
- Une ligne `[intra-save] last.pt écrit` toutes les 3 min
- Une ligne `ep N | train ... val ... SI-SDR ...` à chaque fin d'epoch

Et dans wandb (*Charts → Panel Section*) : courbe `train_loss_step` qui se met à jour en live.

In [ ]:
history = train(run_config, resume_from=resume_from)

## 9. Vérification — checkpoints sur Drive

In [ ]:
ckpt_dir = Path(cfg.CHECKPOINTS) / RUN_ID
log_path = Path(cfg.LOGS) / RUN_ID / "history.jsonl"

print(f"Dossier checkpoints : {ckpt_dir}")
if not ckpt_dir.exists():
    print("  ! ce dossier n'existe pas — aucune sauvegarde n'a été faite.")
else:
    files = sorted(ckpt_dir.iterdir())
    if not files:
        print("  ! dossier vide.")
    else:
        for p in files:
            size_mb = p.stat().st_size / 1e6
            print(f"  {p.name:<20s}  {size_mb:6.1f} MB")
print(f"\nJSONL d'historique : {log_path}")
print("  existe :", log_path.is_file())

## 10. Aperçu rapide

In [ ]:
import matplotlib.pyplot as plt

try:
    history
except NameError:
    from src.logging_utils import read_history, metrics_records
    print("[aperçu] history absente du kernel → lecture depuis le JSONL")
    recs = [r for r in metrics_records(read_history(Path(cfg.LOGS) / RUN_ID))
            if not r.get("is_step")]
    history = {k: [r.get(k) for r in recs]
               for k in ["epoch", "train_loss", "val_loss", "val_si_sdr", "lr", "gap"]}

if not history.get("epoch"):
    print("[aperçu] aucune donnée à afficher — entraînement interrompu trop tôt.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    ep = history["epoch"]

    axes[0, 0].plot(ep, history["train_loss"], label="train")
    axes[0, 0].plot(ep, history["val_loss"],   label="val")
    axes[0, 0].set_title("Loss MSE"); axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(ep, history["val_si_sdr"], color="tab:green")
    axes[0, 1].set_title("SI-SDR val (dB)"); axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(ep, history["lr"], color="tab:orange")
    axes[1, 0].set_yscale("log")
    axes[1, 0].set_title("Learning rate"); axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(ep, history["gap"], color="tab:red")
    axes[1, 1].axhline(0, color="k", linestyle="--", linewidth=0.5)
    axes[1, 1].set_title("Gap (val − train)"); axes[1, 1].grid(alpha=0.3)

    for ax in axes.flatten():
        ax.set_xlabel("epoch")
    fig.tight_layout()
    plt.show()